In [1]:
#package imports
import os
from langchain_google_genai import ChatGoogleGenerativeAI

In [28]:
#loading gemini model with langchain

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=os.getenv("GEMINI_API_KEY"),thinking_budget=0,temperature=0)

In [29]:
class Agent :
    def __init__(self,system=""):
        self.system = system
        self.message = []
        if self.system:
            self.message.append({"role": "system", "content": self.system})

    def __call__(self,message):
        self.message.append({"role": "user", "content": message})
        result = self.execute()
        self.message.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = model.invoke(self.message)
        return completion.content
        
        
        

In [30]:
prompt = '''
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer.
Use Thought to mescribe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate_total_price:
e.g. calculate_total_price: apple: 2, banana: 3
Runs a calculation for the total price based on the quantity and prices of the fruits.

get_fruit_price:
e.g. get_fruit_price: apple
returns the price of the fruit when given its name.

Example session:

Question: Whut is the total price for 2 apples and 3 bananas?
Thought: I should calculate the total price by getting the price of each fruit and summing them up.
Action: get_fruit_price: apple
PAUSE

Observation: The price of an apple is $1.5.

Action: get_fruit_price: banana
PAUSE

Observation: The price of a banana is $1.2.

Action: calculate_total_price: apple: 2, banana: 3
PAUSE

You then output:

Answer: The total price for 2 apples and 3 bananas is $6.6.
'''. strip()


In [31]:
# Price lookup for fruits
fruit_prices = {
"apple": 1.5,
"banana": 1.2,
"orange": 1.3,
"grapes": 2.0}

# Function to calculate the price of a specific fruit
def get_fruit_price(fruit):
    if fruit in fruit_prices:
        return f"The price of a {fruit} is ${fruit_prices[fruit]}"
    else:
        return f"Sorry, I don't know the price of {fruit}."

# Function to calculate total price based on quantities
def calculate_total_price(fruits):
    total = 0.0
    fruit_list = fruits.split(", ")
    for item in fruit_list:
        fruit, quantity = item.split(": ")
        quantity = int (quantity)
        if fruit in fruit_prices:
            total += fruit_prices [fruit] * quantity
        else:
            return f"Sorry, I don't have the price of {fruit}."
    return f"The total price is ${total:.2f}"

# Mapping actions to functions
known_actions = {
"get_fruit_price": get_fruit_price,
"calculate total price": calculate_total_price
}

In [32]:
# Run a query
import re
action_re = re.compile(r'^Action: (\w+): (.*)$') # python regular expression to select action

def query(question):
    bot = Agent(prompt)
    result = bot(question) #_call
    print(result)
    actions = [ action_re.match(a)
                for a in result.split('\n') 
                if action_re.match(a)
              ]
    
    if actions:
        action, action_input = actions[0].groups()
        if action not in known_actions:
            raise Exception(f"Unknown action: {action}: {action_input}")
        print(f" -- running {action} {action_input}")
        observation = known_actions [action] (action_input)
        print("Observation:", observation)
    else:
        return[]

In [33]:
query("what is the total price of 2 apples?")

Thought: I need to find the price of an apple first, then I can calculate the total price for 2 apples.
Action: get_fruit_price: apple
PAUSE
 -- running get_fruit_price apple
Observation: The price of a apple is $1.5
